# Evaluation First — How Do You Know Your RAG Is Good?
**AI Architect · Pillar I · Module 1**

You already know how to *build* RAG. This lab builds the **instrument** that tells you whether it is any good — then baselines a naive RAG. The scorecard you produce is the number every later module has to beat.

This is **eval-driven development**: define *good* first, earn every technique against it afterward.

## 1 · Setup — one key, everything else is bundled

Retrieval is **keyless** (embeddings run locally). Only *generation* and the *LLM-judge* evaluators reach a model, so set **one** LLM key.

In [ ]:
!pip install -q "mai_rag[evals] @ git+https://github.com/balajivis/ai-architect-labs.git"

import os
# In Colab: add a key in the Secrets panel (key icon), then:
try:
    from google.colab import userdata
    for k in ("GROQ_API_KEY", "OPENAI_API_KEY", "GEMINI_API_KEY", "AZURE_OPENAI_API_KEY"):
        try:
            v = userdata.get(k)
            if v:
                os.environ[k] = v
        except Exception:
            pass
except ImportError:
    pass  # running locally: export the key in your shell

## 2 · The data layer is ready to go

`load_policy_corpus()` copies the **pre-embedded** corpus DB into place and returns a ready **SQL + vector** store — instant, no embedding pass. (Pass `rebuild=True` to watch it chunk + embed from the raw docs once.) Query-time search loads MiniLM lazily on the first call.

In [ ]:
from mai_rag import corpus, evals, viz, golden
from mai_rag.baseline import naive_rag

store = corpus.load_policy_corpus("policy.db")   # persisted; re-runs are instant
store.stats()

## 3 · Look at the corpus and a real question

In [ ]:
# The documents in the knowledge base:
for r in store.conn.execute("SELECT title, source FROM documents ORDER BY id"):
    print(f"- {r['title']}  ({r['source']})")

# A candidate question shipped with the corpus:
seed = corpus.load_golden_seed()
print("\nExample golden case:")
print(seed[0]["question"], "->", seed[0]["expected_answer"][:120], "...")

## 4 · Author the golden set

The evaluator is only as good as the cases you point it at. Start from the shipped candidates, then **add your own** — including a multi-hop question that spans two documents, with criteria the answer *must* satisfy.

In [ ]:
gs = golden.GoldenSet.from_seed(store)        # load the candidate cases into the DB

# Author your own — the core skill:
gs.add(golden.GoldenCase(
    question="If an employee loses a laptop with customer data while travelling, "
             "what must they do and within how long?",
    expected="Report to the Security Team within the incident-response notification "
             "window and follow Data Classification handling for customer data.",
    criteria=["report", "Security"],
    tier="blueprint",
    tags=["multi-hop"],
)).save()

print(len(gs), "golden cases;", store.stats()["golden_cases"], "in the DB")

## 5 · Open the box once — what *is* faithfulness?

Before trusting any metric, read it. `faithfulness` decomposes the answer into atomic claims and checks each against the retrieved context. Here is the real implementation — no magic:

In [ ]:
import inspect
from mai_rag.evals import native
print(inspect.getsource(native.faithfulness))

In [ ]:
# Run the baseline on one case and score it by hand:
case = gs.cases[0]
out = naive_rag(store, case.question)
e = evals.EvalInput(question=case.question, answer=out['answer'],
                    contexts=out['contexts'], expected=case.expected, criteria=case.criteria)
print('ANSWER:', out['answer'][:300], '\n')
print(native.faithfulness(e))
print(native.context_precision(e))

## 6 · Trust the suite — score across all three layers

Same shapes as Kapi's nine engines: **retrieval** (context precision/recall) · **generation** (faithfulness, answer-relevancy) · **safety** (PII, relevancy). Swap `backend="ragas"` to run the real RAGAS library instead of the native code — identical output schema.

In [ ]:
for s in evals.evaluate(e):           # backend='ragas' to use the RAGAS library
    print(f"{s.evaluator:18s} {s.score:.2f}  {'PASS' if s.passed else 'FAIL'}  - {s.reasoning[:60]}")

## 7 · Baseline the naive RAG — the number to beat

In [ ]:
run = evals.run_suite(store, gs, naive_rag, label="naive baseline")
viz.scorecard(run["summary"], title="Naive RAG - baseline scorecard")

## 8 · Read the distribution, not the mean

A 0.9 average can hide a catastrophic 0.2 on the cases that matter most. The per-case heatmap surfaces it.

In [ ]:
viz.per_case_heatmap(run["results"])

## The through-line

The baseline now lives in `eval_runs` / `eval_results`. In **Module 2** you add hybrid search + reranking + Contextual Retrieval, then:

```python
viz.compare_runs(store, "naive baseline", "hybrid+rerank")
```

Success is **moving these exact numbers**. Nothing else counts.